# Fase 11 (extensión) — Mapa de Susceptibilidad/Frecuencia de Inundación desde Sentinel-1 (SAR)

**Proyecto:** HCI Ecuador (Hazard Chronicity Index)
**Fase:** 11 (extensión — no forma parte de las 10 fases originales de Fase 0; corresponde a la capa opcional "Sentinel-1 Flood Extent" de Fase 0 §3)
**Autor / Author:** Jerson Achicanoy, Information Management and GIS Expert
**Fecha:** 2026-08-06

## ⚠️ Este notebook NO corre en el entorno principal del proyecto

Todo el resto del proyecto corre en un sandbox sin salida de red hacia
Google Earth Engine (mismo problema ya documentado para las descargas de
HDX/INEC en la Fase 2 y la SST Niño 1+2 en la Fase 5 — verificado
explícitamente: `earthengine.googleapis.com` devuelve `403 Forbidden`
desde ese entorno). **Este notebook debe correrse en Google Colab o en tu
máquina local**, donde sí hay acceso real a GEE.

## Cómo correrlo

1. Sube este notebook **y** `aoi_flood_points.json` (mismo directorio) a
   Colab — deben quedar en la misma carpeta de trabajo (`/content/` por
   defecto en Colab).
2. Ejecuta `!pip install earthengine-api -q` si el import de `ee` falla.
3. Ejecuta las celdas en orden. La primera vez que llames
   `ee.Authenticate(...)` te pedirá iniciar sesión en el navegador — usa la
   cuenta de Google asociada al proyecto `YOUR_GEE_PROJECT_ID`.
4. La celda de exportación lanza 1 tarea (`EFCI_flood_frequency_2015_2025`)
   hacia tu Google Drive (carpeta `HCI_Ecuador_Sentinel1`) y **se queda
   revisando el estado cada 30 segundos, imprimiéndolo en pantalla**, para
   que veas visualmente que algo está pasando (igual que en los scripts de
   otros países) — el enlace de seguimiento manual también queda impreso
   por si prefieres no esperar ahí.
5. Cuando la tarea aparezca como `COMPLETED`, descarga el GeoTIFF de Drive
   y colócalo en `data/raw/sentinel1_flood/` del proyecto — avísame para
   continuar con la clasificación en 5 categorías.

**Versión 1.2 (2026-08-06):** se agregó un diagnóstico previo (Sección
5.1) que revisa cuántas imágenes Sentinel-1 hay disponibles para cada
fecha objetivo antes de procesar nada — esto corrige un error real
(`Image.lt: ... Got 0 and 1`) que ocurría cuando alguna fecha (más
probable en 2015, ver Sección 5) no tenía imágenes disponibles en el AOI.
Las fechas sin cobertura ahora se excluyen automáticamente, con aviso
explícito, en vez de romper el notebook a mitad de camino.

## Metodología (resumen — ver `docs/methodology/11_flood_susceptibility_sar.md` para el detalle completo)

- **Detección de cambio SAR** sobre la banda VH (más sensible que VV a
  superficies de agua en calma) — método UN-SPIDER/Copernicus estándar:
  diferencia en dB entre una imagen "durante inundación" y una imagen de
  referencia "pre-inundación", con umbral fijo de **-3 dB**.
- **15 fechas objetivo**, 3 por cada uno de los 5 episodios de El Niño ya
  documentados en la Fase 5 (2015, 2016, 2017, 2023, 2024) — las fechas de
  mayor número de eventos/severidad tipo "flood" reportadas en la base
  histórica del proyecto, no fechas arbitrarias.
- **AOI**: no todo Ecuador continental — buffer de 5 km sobre los 4,288
  puntos únicos donde históricamente se reportó un evento tipo "flood".
  Evita procesar la Amazonía densa y los Andes de alta pendiente, donde el
  método SAR además es menos confiable.
- **Máscaras**: pendiente > 5° excluida (sombra/layover de radar), agua
  permanente JRC > 90% de ocurrencia excluida (no es inundación nueva),
  limpieza de ruido sal-y-pimienta (mínimo 8 píxeles conectados).
- **Resolución**: cálculo del umbral a 10 m nativos; la capa final
  (frecuencia ya agregada) se remuestrea a 30 m solo al exportar — es ahí
  donde el remuestreo realmente acelera el cómputo/exportación en GEE.
- **Salida**: un GeoTIFF de una banda, valores 0-15 (número de las 15
  fechas en que cada píxel se detectó inundado) — el insumo crudo para la
  clasificación en 5 categorías (se hace después, localmente).

**English:** Sentinel-1 Flood Frequency (Google Earth Engine)

Builds a flood-frequency layer at 30 m from Sentinel-1 VH radar: for each target date it compares the image around the event with a pre-event baseline and flags pixels that darken by more than 3 dB (change detection), masks steep slopes and permanent water, and counts the detections across dates.

Requires a Google Earth Engine account. Replace `YOUR_GEE_PROJECT_ID` with your own project. The classified raster is produced by the SAR classification notebook.

## 0. Autenticación (snippet exacto del proyecto — no cambiar sin justificarlo)

**English:** 0. Authentication

In [ ]:
# Google Earth Engine Python API
import ee

# Authenticate with Earth Engine through a local browser flow
ee.Authenticate(auth_mode="localhost")
# Initialise Earth Engine with your own cloud project ID (no credentials are stored in this repository)
ee.Initialize(project="YOUR_GEE_PROJECT_ID")  # replace with your own Google Earth Engine cloud project

## 1. Parámetros ajustables

Todo lo que alguien querría cambiar vive aquí, no enterrado en el resto
del notebook.

**English:** 1. Adjustable parameters

In [ ]:
# Buffer around each historical flood point, in metres
BUFFER_AOI_M = 5000            # buffer alrededor de cada punto de evento histórico
# VH backscatter drop (dB) that marks a pixel as flooded (UN-SPIDER standard)
THRESHOLD_DB = -3.0            # umbral de caída de backscatter VH para "inundado" (UN-SPIDER estándar)
# Maximum terrain slope in degrees; steeper pixels are excluded
SLOPE_MAX_DEG = 5              # pendiente máxima permitida (excluye sombra/layover de radar)
# Pixels with historical water occurrence above this % are excluded as permanent water
JRC_WATER_OCCURRENCE_MAX = 90  # % de ocurrencia histórica de agua -> se excluye como "agua permanente"
# Minimum size of connected flooded patches at the original resolution
CONNECTED_PIXEL_MIN = 8        # limpieza de ruido sal-y-pimienta
# Export resolution in metres
EXPORT_SCALE_M = 30            # resolución del producto final (frecuencia agregada)
# Length of the pre-event baseline window, in days
BASELINE_LOOKBACK_DAYS = 36    # ventana hacia atrás para construir la imagen de referencia (~3 ciclos S1)
# Gap in days between the event date and the start of the baseline window
BASELINE_GAP_DAYS = 12         # días de "colchón" entre la fecha evento y el inicio de la ventana de referencia
# +/- days around the target date when searching for the 'during event' image
EVENT_WINDOW_DAYS = 6          # +/- días alrededor de la fecha objetivo para buscar la imagen "durante evento"

# Google Drive folder that receives the export
DRIVE_FOLDER = "HCI_Ecuador_Sentinel1"

## 2. Fechas objetivo

3 por cada uno de los 5 episodios de El Niño ya documentados en la Fase 5
del proyecto (`docs/methodology/05_recurrence_index.md`), elegidas por ser
las de mayor número de eventos + severidad tipo "flood" reportados en la
base histórica SNGRE/ECU911 dentro de cada episodio (cálculo verificado
localmente antes de escribir este notebook, no fechas arbitrarias).

**English:** 2. Target dates

In [ ]:
# Target dates grouped by El Nino episode
FECHAS_OBJETIVO = {
    "2015": ["2015-03-20", "2015-04-26", "2015-02-07"],
    "2016": ["2016-01-25", "2016-03-06", "2016-03-03"],
    "2017": ["2017-04-08", "2017-03-24", "2017-02-08"],
    "2023": ["2023-04-18", "2023-03-09", "2023-03-23"],
    "2024": ["2024-02-21", "2024-02-20", "2024-02-19"],
}
# Flat list of all target dates
TODAS_LAS_FECHAS = [f for lista in FECHAS_OBJETIVO.values() for f in lista]
# Check there are 15 dates (3 per episode, 5 episodes)
assert len(TODAS_LAS_FECHAS) == 15, "se esperaban 15 fechas objetivo (3 x 5 episodios)"
# Print the number of target dates
print(f"{len(TODAS_LAS_FECHAS)} fechas objetivo cargadas.")

## 3. AOI: puntos históricos de eventos "flood"

Unión de buffers de 5 km alrededor de los puntos históricos de eventos
tipo "flood" (4,288 puntos únicos, redondeados a ~1 km, generados
localmente desde `hazard_event_geo.gpkg` — ver `aoi_flood_points.json`).

**Nota sobre la ruta del archivo (corregido 2026-08-06):** la primera
versión de este código (como script `.py`) usaba `Path(__file__).parent`
para ubicar `aoi_flood_points.json` junto al script — eso falla en un
notebook/Colab con `NameError: name '__file__' is not defined`, porque
una celda de notebook no tiene la noción de "archivo actual" que sí tiene
un script ejecutado con `python archivo.py`. La celda de abajo intenta
`Path(__file__)` primero (por si en algún momento se ejecuta como script)
y, si falla, usa el directorio de trabajo actual (`Path.cwd()`) — que en
Colab es donde subiste el notebook y el JSON juntos.

**English:** 3. Area of interest: historical flood event points

In [ ]:
# JSON reader
import json
# Object-oriented file paths
from pathlib import Path

# Try the standard quantile cut
try:
    # Folder of the notebook when run as a script
    base_dir = Path(__file__).parent
except NameError:
    # Normal en un notebook/Colab: no hay '__file__', se usa el directorio
    # de trabajo actual — asegúrate de haber subido aoi_flood_points.json
    # a esa misma carpeta (en Colab, el panel de archivos a la izquierda).
    # Folder of the notebook when run interactively
    base_dir = Path.cwd()

# Path to the file with the historical flood points
aoi_path = base_dir / "aoi_flood_points.json"
# Check the points file exists
assert aoi_path.exists(), (
    f"No se encontró {aoi_path}. Sube 'aoi_flood_points.json' a la misma "
    "carpeta que este notebook (en Colab: panel de archivos a la izquierda, "
    "arrastrar el archivo a /content/)."
)

# Open the points file
with aoi_path.open("r", encoding="utf-8") as f:
    # Read the points
    aoi_data = json.load(f)

# List of (longitude, latitude) pairs
puntos = aoi_data["puntos_lonlat"]  # lista de [lon, lat]
# Print the number of points
print(f"Puntos históricos de eventos 'flood' cargados: {len(puntos)}")

# Earth Engine multipoint geometry
geom_puntos = ee.Geometry.MultiPoint(puntos)
# Area of interest: buffer around the flood points; bounds() gives a simple geometry usable as region
AOI = geom_puntos.buffer(BUFFER_AOI_M).bounds()  # bounds() simplifica la geometría para uso como 'region' de exportación
# Union of the buffers around the points
AOI_UNION = geom_puntos.buffer(BUFFER_AOI_M)  # geometría real (no el bounding box) para enmascarar

## 4. Capas ancilares: pendiente (DEM) y agua permanente (JRC)

**English:** 4. Ancillary layers: slope (DEM) and permanent water (JRC)

In [ ]:
# Digital elevation model (SRTM)
dem = ee.Image("USGS/SRTMGL1_003")
# Slope in degrees
slope = ee.Terrain.slope(dem)
# Mask of pixels with gentle slopes
mascara_pendiente = slope.lte(SLOPE_MAX_DEG)

# JRC Global Surface Water dataset
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")
# Mask excluding permanent water
mascara_agua_permanente = jrc.select("occurrence").lt(JRC_WATER_OCCURRENCE_MAX).unmask(1)

# Combined mask clipped to the area of interest
mascara_base = mascara_pendiente.And(mascara_agua_permanente).clip(AOI_UNION)
# Print that the masks are ready
print("Máscaras de pendiente y agua permanente listas.")

## 5. Función: máscara binaria de inundación por fecha

Dado un día objetivo, construye la máscara binaria de inundación
(1 = inundado, 0 = no) usando cambio VH.

**Corregido 2026-08-06 — error real encontrado al ejecutar:**
`Image.lt: If one image has no bands, the other must also have no bands.
Got 0 and 1.` Causa: para algunas fechas, `col_evento` o `col_referencia`
quedan **vacías** (0 imágenes Sentinel-1 disponibles en esa ventana sobre
el AOI) — `.median()` de una colección vacía devuelve una imagen de 0
bandas, y restar/comparar eso contra una constante numérica revienta con
ese error exacto. Esto es más probable en fechas de 2015 (Sentinel-1A
apenas llevaba ~8-11 meses operando cuando ocurrieron los episodios más
tempranos del proyecto, y la cobertura sistemática sobre Sudamérica no
era tan consistente como en años posteriores) y, dado que el AOI de este
proyecto cubre una fracción considerable de Ecuador (~100,000-150,000
km², ver `docs/methodology/11_flood_susceptibility_sar.md` §3.2), es
plausible que falte cobertura en alguna sub-zona para ciertas fechas
puntuales. La Sección 5.1 diagnostica esto explícitamente ANTES de
construir ninguna máscara, y la función de abajo además queda blindada
por si acaso.

**English:** 5. Per-date flood mask function

In [ ]:
# Sentinel-1 VH collection between two dates
def coleccion_s1_vh(fecha_ini, fecha_fin):
    """Colección Sentinel-1 GRD, IW, VH, dentro del AOI y rango de fechas.

    English:
    Sentinel-1 VH collection for a date range, filtered to the area of interest.

    Args:
        fecha_ini: Start date.
        fecha_fin: End date.

    Returns:
        Earth Engine image collection.
    """
    # Weighted sum of the four components
    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(AOI_UNION)
        .filterDate(fecha_ini, fecha_fin)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VH"))
        .select("VH")
    )


# Compute the event and baseline windows for a target date
def ventanas_fecha(fecha_str):
    """Calcula las ventanas de fechas 'evento' y 'referencia' para una fecha objetivo.

    Args:
        fecha_str: fecha objetivo en formato 'YYYY-MM-DD'.

    Returns:
        Tupla (ee.Date evento_ini, ee.Date evento_fin, ee.Date ref_ini, ee.Date ref_fin).

    English:
    Compute the event window and the baseline window for a target date.

    Args:
        fecha_str: Target date (YYYY-MM-DD).

    Returns:
        Event start, event end, baseline start and baseline end.
    """
    # Target date as an Earth Engine date
    fecha = ee.Date(fecha_str)
    # Start of the event window
    evento_ini = fecha.advance(-EVENT_WINDOW_DAYS, "day")
    # End of the event window
    evento_fin = fecha.advance(EVENT_WINDOW_DAYS, "day")
    # End of the baseline window
    ref_fin = fecha.advance(-BASELINE_GAP_DAYS, "day")
    # Start of the baseline window
    ref_ini = ref_fin.advance(-BASELINE_LOOKBACK_DAYS, "day")
    # Return the four dates
    return evento_ini, evento_fin, ref_ini, ref_fin


# Binary flood mask for one target date
def mascara_inundacion_fecha(fecha_str):
    """Genera la máscara binaria de inundación (0/1) para una fecha objetivo.

    Args:
        fecha_str: fecha objetivo en formato 'YYYY-MM-DD' (fecha reportada
            de mayor severidad/n. de eventos dentro de un episodio El Niño).
            Debe ser una fecha ya verificada como 'con datos' en la Sección
            5.1 — esta función no vuelve a chequear disponibilidad, solo
            se blinda contra el caso 0-bandas por seguridad adicional.

    Returns:
        ee.Image de una banda ('flood'), valores 0/1, recortada al AOI.

    English:
    Build the binary flood mask for one target date by change detection.

    Args:
        fecha_str: Target date (YYYY-MM-DD).

    Returns:
        Image with 1 for flooded pixels and 0 elsewhere.
    """
    # Event and baseline windows
    # Event and baseline date windows
    evento_ini, evento_fin, ref_ini, ref_fin = ventanas_fecha(fecha_str)

    # Images around the event
    # Sentinel-1 VH collection during the event window
    col_evento = coleccion_s1_vh(evento_ini, evento_fin)
    # Images in the baseline period
    # Sentinel-1 VH collection during the baseline window
    col_referencia = coleccion_s1_vh(ref_ini, ref_fin)

    # Blindaje contra colecciones vacías (0 bandas): si por algún motivo
    # llega aquí una fecha sin imágenes, se sustituye por una imagen
    # constante fuera de rango (no dispara el umbral de inundación) en vez
    # de dejar que .median() de una colección vacía rompa la resta/umbral
    # más abajo con el error "Got 0 and 1" ya documentado arriba.
    # Median image of the event period
    # Median event image (constant 0 if the collection is empty)
    img_evento = ee.Image(ee.Algorithms.If(
        col_evento.size().gt(0), col_evento.median(), ee.Image.constant(0).rename("VH")
    ))
    # Median image of the baseline period
    # Median baseline image (constant 0 if the collection is empty)
    img_referencia = ee.Image(ee.Algorithms.If(
        col_referencia.size().gt(0), col_referencia.median(), ee.Image.constant(0).rename("VH")
    ))

    # Diferencia en dB (ambas bandas ya vienen en dB desde el catálogo GRD de GEE)
    # Difference in backscatter (dB) between event and baseline
    # Difference in dB: event minus baseline
    diferencia = img_evento.subtract(img_referencia).rename("dif_db")

    # Flag pixels that darkened beyond the threshold
    inundado = diferencia.lt(THRESHOLD_DB)
    # Apply the slope and permanent-water mask
    inundado = inundado.updateMask(mascara_base)

    # Si a esta fecha le faltó cualquiera de las dos colecciones, se
    # enmascara por completo — "sin observación válida" no debe contarse
    # como "no inundado" (sesgaría la frecuencia hacia abajo).
    # Check both collections contain images
    hay_datos = col_evento.size().gt(0).And(col_referencia.size().gt(0))
    # Mask everything when data is missing
    inundado = inundado.updateMask(ee.Image.constant(hay_datos))

    # Limpieza de ruido sal-y-pimienta: exige un mínimo de píxeles conectados
    # Count connected flooded pixels
    conteo_conectado = inundado.connectedPixelCount(25, True)
    # Keep only flooded areas made of enough connected pixels (removes speckle)
    inundado_limpio = inundado.updateMask(conteo_conectado.gte(CONNECTED_PIXEL_MIN))

    # Vuelve a 0/1 explícito (los píxeles enmascarados no cuentan en la suma
    # posterior, que es justamente lo que queremos: "sin observación válida"
    # es distinto de "observado, no inundado")
    # Return the final mask for the date
    return inundado_limpio.unmask(0).rename("flood").clip(AOI_UNION).set("fecha", fecha_str)


# Print that the functions are defined
print("Funciones de detección de cambio definidas.")

## 5.1 Diagnóstico de disponibilidad de imágenes (nuevo, 2026-08-06)

Antes de construir ninguna máscara, se consulta cuántas imágenes
Sentinel-1 (evento y referencia) hay realmente disponibles para cada una
de las 15 fechas objetivo — esto es exactamente lo que causó el error
`Image.lt: ... Got 0 and 1` la primera vez: una fecha con 0 imágenes en
alguna de las dos ventanas. Corriendo este diagnóstico primero, **se ve
explícitamente qué fechas hay que excluir**, en vez de que el notebook
falle a mitad de camino sin decir por qué.

Esto es también lo que permite "ver visualmente qué está pasando" antes
de lanzar nada pesado — cada fecha imprime su conteo apenas se calcula.

**English:** 5.1 Image availability diagnostic

In [ ]:
# Print a header for the availability check
print("Verificando disponibilidad de Sentinel-1 por fecha objetivo (evento AOI: Ecuador, zonas historicamente inundables)...\n")

# Dictionary of availability per date
disponibilidad = {}
# Loop over the target dates
for f in TODAS_LAS_FECHAS:
    # Event and baseline windows
    evento_ini, evento_fin, ref_ini, ref_fin = ventanas_fecha(f)
    # Number of event images
    n_evento = coleccion_s1_vh(evento_ini, evento_fin).size().getInfo()
    # Number of baseline images
    n_referencia = coleccion_s1_vh(ref_ini, ref_fin).size().getInfo()
    # Date is usable only with images in both periods
    ok = n_evento > 0 and n_referencia > 0
    # Store the result
    disponibilidad[f] = ok
    # Status label
    estado = "OK" if ok else "SIN DATOS -> se excluye"
    # Print the image counts per date
    print(f"  {f}: {n_evento:2d} imagenes evento, {n_referencia:2d} imagenes referencia  [{estado}]")

# Dates with coverage
FECHAS_VALIDAS = [f for f, ok in disponibilidad.items() if ok]
# Dates without coverage
FECHAS_EXCLUIDAS = [f for f, ok in disponibilidad.items() if not ok]

# Print how many dates have coverage
print(f"\n{len(FECHAS_VALIDAS)} de {len(TODAS_LAS_FECHAS)} fechas objetivo tienen cobertura Sentinel-1 completa.")
# Report excluded dates, if any
if FECHAS_EXCLUIDAS:
    # Print the excluded dates
    print(f"Excluidas por falta de cobertura: {FECHAS_EXCLUIDAS}")
    # Note that the maximum frequency equals the number of valid dates
    print("(Documentar esto en el reporte final: el techo real de la capa de")
    # Print the maximum possible frequency
    print(f" frecuencia es {len(FECHAS_VALIDAS)}, no {len(TODAS_LAS_FECHAS)}, si alguna fecha quedo excluida.)")

## 5.2 Diagnóstico: umbral Otsu (automático) vs. umbral fijo -3 dB (nuevo, 2026-08-06)

El pipeline usa un umbral **fijo** de -3 dB (estándar UN-SPIDER, elegido por
ser reproducible entre fechas y robusto en un AOI heterogéneo — ver
`docs/methodology/11_flood_susceptibility_sar.md` §3.3). Esta celda **no
cambia ese umbral operativo**, solo lo valida: calcula, para cada fecha
válida, el histograma real de la diferencia VH (evento - referencia) dentro
del AOI y deriva el umbral que separaría automáticamente los dos grupos del
histograma (método de Otsu — el mismo usado en visión por computador para
binarizar imágenes con dos modas). Si el Otsu de cada fecha ronda -3 dB,
el umbral fijo está bien calibrado para estos datos. Si se aleja mucho
(por ejemplo, consistentemente en -5 o -1 dB), es señal de que el umbral
fijo debería reconsiderarse.

El histograma se calcula a 100 m (más grueso que el resto del análisis,
solo para este diagnóstico) para que la consulta a GEE sea rápida — no
afecta el cálculo operativo, que sigue siendo a 10 m nativo.

**English:** See the Spanish text above. Section: 5.2 Diagnóstico: umbral Otsu (automático) vs. umbral fijo -3 dB (nuevo, 2026-08-06).

In [ ]:
# VH difference (event minus baseline) for one date, before thresholding
def diferencia_db_fecha(fecha_str):
    """Recalcula solo la diferencia VH (evento - referencia) para una fecha,
    sin aplicar todavía ningún umbral -- es la misma resta que hace
    mascara_inundacion_fecha() internamente, expuesta aparte para poder
    inspeccionar su histograma real antes de decidir/confirmar el umbral.

    Args:
        fecha_str: fecha objetivo en formato 'YYYY-MM-DD', debe estar en
            FECHAS_VALIDAS (con cobertura Sentinel-1 confirmada en 5.1).

    Returns:
        ee.Image de una banda ('dif_db'), recortada al AOI y a la máscara
        base (pendiente + agua permanente), sin umbralizar.

    English:
    Recompute only the VH difference (event minus reference) for one date, without applying any threshold. Same subtraction as the flood-mask function, exposed separately so its real histogram can be inspected before the threshold is confirmed. Args: fecha_str: target date 'YYYY-MM-DD' (must be in the valid dates with confirmed Sentinel-1 coverage). Returns: single-band ee.Image ('dif_db') clipped to the AOI and the base mask (slope + permanent water), not thresholded.
    """
    evento_ini, evento_fin, ref_ini, ref_fin = ventanas_fecha(fecha_str)
    col_evento = coleccion_s1_vh(evento_ini, evento_fin)
    col_referencia = coleccion_s1_vh(ref_ini, ref_fin)
    img_evento = ee.Image(ee.Algorithms.If(
        col_evento.size().gt(0), col_evento.median(), ee.Image.constant(0).rename("VH")
    ))
    img_referencia = ee.Image(ee.Algorithms.If(
        col_referencia.size().gt(0), col_referencia.median(), ee.Image.constant(0).rename("VH")
    ))
    diferencia = img_evento.subtract(img_referencia).rename("dif_db")
    # Apply the base mask and clip to the AOI
    return diferencia.updateMask(mascara_base).clip(AOI_UNION)


# Otsu threshold (maximises between-class variance) from a histogram fetched with getInfo()
def otsu_desde_histograma(histograma):
    """Calcula el umbral Otsu (maximiza la varianza entre clases) a partir
    de un histograma de GEE ya traído a Python via .getInfo().

    Args:
        histograma: diccionario con 'bucketMeans' e 'histogram' (o
            'bucketMin'/'bucketWidth' según la version de GEE) tal como lo
            devuelve ee.Reducer.histogram().

    Returns:
        Umbral Otsu (float, en dB) que maximiza la separacion entre las dos
        clases del histograma (sin inundacion / inundacion).
    """
    # Bin counts
    counts = np.array(histograma["histogram"], dtype=float)
    if "bucketMeans" in histograma:
        # Bin centres
        means = np.array(histograma["bucketMeans"], dtype=float)
    else:
        # Rebuild bin centres from min and width
        ancho = histograma["bucketWidth"]
        means = histograma["bucketMin"] + ancho * (np.arange(len(counts)) + 0.5)

    # Total number of pixels
    total = counts.sum()
    # Empty histogram: no threshold
    if total == 0:
        return None
    # Weighted sum of all values
    suma_total = float((counts * means).sum())

    # Running sums for the background class
    suma_b, peso_b = 0.0, 0.0
    # Best between-class variance and its threshold so far
    mejor_varianza, mejor_umbral = -1.0, float(means[0])
    # Try every bin edge as a candidate threshold
    for i in range(len(counts)):
        peso_b += counts[i]
        if peso_b == 0:
            continue
        peso_f = total - peso_b
        if peso_f == 0:
            break
        suma_b += counts[i] * means[i]
        media_b = suma_b / peso_b
        media_f = (suma_total - suma_b) / peso_f
        # Between-class variance of this candidate
        varianza_entre = peso_b * peso_f * (media_b - media_f) ** 2
        if varianza_entre > mejor_varianza:
            mejor_varianza = varianza_entre
            mejor_umbral = float(means[i])
    # Return the best threshold (dB)
    return mejor_umbral

In [ ]:
# Numerical arrays
import numpy as np

# Coarser scale than the operational analysis, only for this diagnostic histogram
ESCALA_DIAGNOSTICO_M = 100  # mas grueso que el analisis operativo, solo para este histograma

print("Comparando umbral Otsu (automatico, por fecha) vs. umbral fijo operativo "
      f"({THRESHOLD_DB} dB)...\n")

# Otsu threshold per date
resultados_otsu = {}
# Loop over the dates with confirmed coverage
for f in FECHAS_VALIDAS:
    # VH difference for this date
    dif = diferencia_db_fecha(f)
    # Histogram of the difference over the AOI
    hist_dict = dif.reduceRegion(
        reducer=ee.Reducer.histogram(maxBuckets=100),
        geometry=AOI_UNION,
        scale=ESCALA_DIAGNOSTICO_M,
        maxPixels=1e10,
        bestEffort=True,
    ).get("dif_db")
    # Bring the histogram to Python
    hist_info = hist_dict.getInfo()
    if hist_info is None:
        print(f"  {f}: sin histograma valido (revisar cobertura) -- omitido")
        continue
    # Automatic Otsu threshold for this date
    umbral_otsu = otsu_desde_histograma(hist_info)
    resultados_otsu[f] = umbral_otsu
    # Gap between Otsu and the fixed operational threshold
    diferencia_vs_fijo = umbral_otsu - THRESHOLD_DB
    print(f"  {f}: Otsu = {umbral_otsu:.2f} dB   (fijo = {THRESHOLD_DB} dB, "
          f"diferencia = {diferencia_vs_fijo:+.2f} dB)")

# Summary across dates
if resultados_otsu:
    valores_otsu = list(resultados_otsu.values())
    # Average Otsu threshold
    promedio_otsu = sum(valores_otsu) / len(valores_otsu)
    print(f"\nPromedio Otsu sobre {len(valores_otsu)} fechas: {promedio_otsu:.2f} dB "
          f"(fijo operativo: {THRESHOLD_DB} dB)")
    print("Si el promedio Otsu esta cerca de -3 dB, el umbral fijo esta bien calibrado "
          "para este AOI/periodo. Una desviacion grande y consistente (>1-1.5 dB) en la "
          "misma direccion en la mayoria de las fechas sugeriria ajustar THRESHOLD_DB.")
    print("\nRecordatorio: este diagnostico NO cambia el umbral operativo -3 dB usado "
          "en la Seccion 6 -- es solo una validacion. Si se decide ajustar THRESHOLD_DB "
          "tras revisar este resultado, hacerlo en la Seccion 1 (Parametros) y volver a "
          "correr desde ahi.")

## 5.3 Diagnóstico: ¿cuánta área de más marcaría el umbral Otsu? (nuevo, 2026-08-06)

El §5.2 mostró que en 2017 y 2023 el umbral Otsu (-0.46 a +0.25 dB) se
aleja mucho del fijo (-3 dB), mientras que en 2024 coinciden bien. La
hipótesis de trabajo es que Otsu falla aquí por desbalance de clases (los
píxeles realmente inundados son una fracción minúscula del total, y Otsu
no maneja bien histogramas tan desbalanceados) — no que el umbral fijo
esté mal calibrado. Esta celda pone esa hipótesis a prueba de forma
directa: calcula, para cada fecha, cuánta área quedaría marcada como
"inundada" usando el umbral fijo (-3 dB) vs. usando el umbral Otsu de esa
misma fecha. Si Otsu infla el área varias veces sobre el umbral fijo, es
evidencia de que está capturando ruido/speckle general (variación normal
de backscatter), no agua nueva — confirmando que el umbral fijo es la
opción más segura, no el Otsu.

**English:** See the Spanish text above. Section: 5.3 Diagnóstico: ¿cuánta área de más marcaría el umbral Otsu? (nuevo, 2026-08-06).

In [ ]:
print(f"{'Fecha':<12} {'Área -3dB fijo (km²)':>22} {'Área Otsu (km²)':>18} {'Razón Otsu/fijo':>18}")

total_fijo, total_otsu = 0.0, 0.0
for f in FECHAS_VALIDAS:
    if f not in resultados_otsu or resultados_otsu[f] is None:
        continue
    dif = diferencia_db_fecha(f)
    umbral_otsu_f = resultados_otsu[f]

    # Flood mask with the fixed threshold
    mascara_fijo = dif.lt(THRESHOLD_DB)
    # Flood mask with the Otsu threshold
    mascara_otsu = dif.lt(umbral_otsu_f)

    # Flooded area (m²) with the fixed threshold
    area_fijo_m2 = mascara_fijo.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=AOI_UNION, scale=ESCALA_DIAGNOSTICO_M,
        maxPixels=1e10, bestEffort=True,
    ).get("dif_db").getInfo() or 0.0
    # Flooded area (m²) with the Otsu threshold
    area_otsu_m2 = mascara_otsu.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=AOI_UNION, scale=ESCALA_DIAGNOSTICO_M,
        maxPixels=1e10, bestEffort=True,
    ).get("dif_db").getInfo() or 0.0

    # Convert to km²
    area_fijo_km2 = area_fijo_m2 / 1e6
    area_otsu_km2 = area_otsu_m2 / 1e6
    # Ratio of Otsu area to fixed-threshold area
    razon = (area_otsu_km2 / area_fijo_km2) if area_fijo_km2 > 0 else float("nan")

    # Accumulate totals (simple sum, not a union)
    total_fijo += area_fijo_km2
    total_otsu += area_otsu_km2

    print(f"{f:<12} {area_fijo_km2:>22,.1f} {area_otsu_km2:>18,.1f} {razon:>17.1f}x")

print(f"\n{'TOTAL (suma simple, no unión)':<12} {total_fijo:>22,.1f} {total_otsu:>18,.1f} "
      f"{(total_otsu/total_fijo if total_fijo>0 else float('nan')):>17.1f}x")
print("\nNota: 'Área' aquí es una detección por fecha individual (a la escala de "
      f"diagnóstico de {ESCALA_DIAGNOSTICO_M} m), no la capa de frecuencia final -- "
      "sirve solo para comparar el efecto relativo de cada umbral, no reemplaza el "
      "cálculo operativo (que sigue siendo a 10 m nativo, Sección 6).")

## 6. Genera las máscaras (solo fechas con datos) y las agrega en una capa de frecuencia

**English:** 6. Build the masks (dates with data only) and aggregate into a frequency layer

In [ ]:
# Check at least one date has coverage
assert len(FECHAS_VALIDAS) > 0, "ninguna fecha objetivo tiene cobertura Sentinel-1 -- revisar el AOI o las fechas antes de continuar"

# Flood mask for every valid date
mascaras = [mascara_inundacion_fecha(f) for f in FECHAS_VALIDAS]
# Collection of masks
coleccion_mascaras = ee.ImageCollection(mascaras)

# El reducer .sum() de GEE promueve el tipo de pixel a "Long" (64 bits) por
# seguridad ante overflow, sin importar que las mascaras de entrada sean
# 0/1 -- Export.image.toDrive no acepta "Long" (solo tipos flotantes o
# enteros mas pequenos), por eso se convierte explicitamente con
# .toUint8(): el rango real es 0 a 15 (numero de fechas validas), que cabe
# de sobra en un entero sin signo de 8 bits.
# Frequency layer: number of dates each pixel was flagged as flooded
frecuencia = coleccion_mascaras.sum().toUint8().rename("frecuencia_inundacion")
frecuencia = frecuencia.reproject(crs="EPSG:4326", scale=10)  # cálculo a resolución nativa...
# Frequency layer at the export grid
frecuencia_export = frecuencia.reproject(crs="EPSG:4326", scale=EXPORT_SCALE_M).toUint8()  # ...remuestreado solo para exportar (reproject puede volver a promover el tipo, se re-castea por seguridad)

# Print the number of dates behind the layer
print(f"Capa de frecuencia construida sobre {len(FECHAS_VALIDAS)} fechas validas "
      f"(de {len(TODAS_LAS_FECHAS)} objetivo): valores posibles 0 a {len(FECHAS_VALIDAS)}.")

## 7. Exportación a Google Drive

**Nuevo (2026-08-06):** además de dejar el enlace de seguimiento manual,
la celda revisa el estado de la tarea **cada 30 segundos** y lo imprime
—mismo comportamiento que tenías en los scripts de otros países— para
poder ver visualmente que algo está pasando en vez de una celda "colgada"
sin retroalimentación. Se puede interrumpir la celda en cualquier momento
(el botón de detener de Jupyter/Colab) sin cancelar la tarea en GEE — la
tarea sigue corriendo del lado de Google aunque cierres el notebook; solo
se detiene el seguimiento visual local.

**English:** 7. Export to Google Drive

In [ ]:
# Timing utilities
import time

# Define the export task to Google Drive
tarea_frecuencia = ee.batch.Export.image.toDrive(
    image=frecuencia_export,
    description="EFCI_flood_frequency_2015_2025",
    folder=DRIVE_FOLDER,
    fileNamePrefix="efci_flood_frequency_2015_2025",
    region=AOI,
    scale=EXPORT_SCALE_M,
    crs="EPSG:4326",
    maxPixels=1e10,
)
# Start the export
tarea_frecuencia.start()

print(f"Tarea de exportación iniciada: EFCI_flood_frequency_2015_2025 (id: {tarea_frecuencia.id})")
# Print where to track the task
print("Seguimiento manual (si prefieres no esperar aquí): https://code.earthengine.google.com/tasks\n")

# Polling interval in seconds
INTERVALO_SEGUNDOS = 30
# Wait while the task is running
while tarea_frecuencia.active():
    # Current task status
    estado = tarea_frecuencia.status()
    # Current time
    marca_tiempo = time.strftime("%H:%M:%S")
    # Print the status
    print(f"[{marca_tiempo}] Estado: {estado['state']}")
    # Wait before checking again
    time.sleep(INTERVALO_SEGUNDOS)

# Final task status
estado_final = tarea_frecuencia.status()
# Print the final status
print(f"\nTarea finalizada con estado: {estado_final['state']}")
# Check whether the task failed
if estado_final["state"] == "FAILED":
    # Print the Earth Engine error message
    print("Mensaje de error de GEE:", estado_final.get("error_message", "(sin detalle)"))
    # Print a hint about missing coverage
    print("Revisa el diagnostico de la Seccion 5.1 -- si el error menciona bandas o")
    # Print a second hint
    print("imagenes faltantes, probablemente falte cobertura Sentinel-1 en alguna fecha/zona del AOI.")
else:
    # Print where to download the GeoTIFF
    print("Descarga el GeoTIFF desde Google Drive, carpeta:", DRIVE_FOLDER)

## 8. (Opcional) Exportación de las 15 máscaras individuales

Para control de calidad — verificar visualmente que cada fecha capturó
agua real, no ruido. Comentado por defecto para no multiplicar tareas de
exportación innecesariamente; descomenta si necesitas depurar una fecha
específica.

**English:** 8. (Optional) Export the individual masks

In [ ]:
# for img in mascaras:
#     fecha = img.get("fecha").getInfo()
#     ee.batch.Export.image.toDrive(
#         image=img, description=f"flood_mask_{fecha}", folder=DRIVE_FOLDER,
#         fileNamePrefix=f"flood_mask_{fecha}", region=AOI, scale=10,
#         crs="EPSG:4326", maxPixels=1e10,
#     ).start()

## 9. Siguientes pasos

Cuando la tarea `EFCI_flood_frequency_2015_2025` aparezca como
`COMPLETED` en <https://code.earthengine.google.com/tasks>:

1. Descarga el GeoTIFF desde tu Google Drive (carpeta `HCI_Ecuador_Sentinel1`).
2. Colócalo en `data/raw/sentinel1_flood/` del proyecto HCI Ecuador.
3. Avisa para continuar con la clasificación en 5 categorías (Jenks) y su
   integración al resto del proyecto.

**English:** See the Spanish text above. Section: 9. Siguientes pasos.